# Local Ollama model tests

Start Ollama and pull the model you want to test (`ollama pull <model>`).

Select this project's Python environment (requests, pandas, IPython) and run cells top to bottom.
Edit `BASE_URL` and `MODEL_NAMES` below, or set `OLLAMA_BASE_URL`, `OLLAMA_MODEL`, and optionally `OLLAMA_API_KEY` in the kernel environment. Model defaults are retained from the original notebooks; use the server's advertised IDs.

Each model receives the same five prompts and generation settings. These are sequential smoke benchmarks, not a concurrent load test. The first test request warms the model before measurement; loading another Ollama model can still affect its first benchmark request.

Chat requests use `/v1/chat/completions` so the server applies its chat template.

In [1]:
import os
import time
from collections import Counter

import pandas as pd
import requests
from IPython.display import display

BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
MODEL_NAMES = [os.environ.get("OLLAMA_MODEL", "gemma4:26b")]
# Add other installed/served model IDs to MODEL_NAMES to compare them.
API_KEY = os.environ.get("OLLAMA_API_KEY", "")
MAX_TOKENS = 800
TEMPERATURE = 0.7
NUM_REQUESTS = 5
REQUEST_TIMEOUT = (10, 180)  # Connection timeout, then inference timeout.
BENCHMARK_PROMPTS = [
    "What is the capital of France?",
    "Write a short poem about artificial intelligence.",
    "Explain quantum computing in simple terms.",
    "What are the three laws of robotics?",
    "Summarize the plot of the movie Inception.",
]
headers = {"Content-Type": "application/json"}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"
CHAT_URL = f"{BASE_URL}/v1/chat/completions"
print(f"Endpoint: {CHAT_URL}")
print(f"Models: {MODEL_NAMES}")

Endpoint: http://localhost:11434/v1/chat/completions
Models: ['gemma4:26b']


## Discover served models
Run this cell before testing; choose model IDs from its output.

In [ ]:
# List the actual model IDs advertised by the inference server.
response = requests.get(
    f"{BASE_URL}/v1/models",
    headers=headers,
    timeout=REQUEST_TIMEOUT,
)
response.raise_for_status()
available_models = [item["id"] for item in response.json()["data"]]
print("Available models:", available_models)
missing_models = set(MODEL_NAMES) - set(available_models)
if missing_models:
    raise ValueError(
        f"Models not served: {sorted(missing_models)}. "
        "Update MODEL_NAMES using the IDs listed above and rerun setup."
    )

Available models: ['gemma4:26b']


## Request helper and smoke test

In [ ]:
def chat(prompt, model):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
        "stream": False,
    }
    start = time.perf_counter()
    response = requests.post(
        CHAT_URL,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()
    elapsed = time.perf_counter() - start
    choices = data.get("choices")
    if not isinstance(choices, list) or not choices:
        raise ValueError("Response has no chat completion choices")
    choice = choices[0]
    message = choice.get("message") or {}
    text = message.get("content") or ""
    if not isinstance(text, str):
        raise ValueError("Expected string message.content")
    reasoning = message.get("reasoning_content") or message.get("reasoning") or ""
    if not text.strip() and not reasoning:
        raise ValueError("Model returned no text or reasoning")
    usage = data.get("usage") or {}
    completion_tokens = usage.get("completion_tokens")
    return {
        "model": model,
        "served_model": data.get("model", model),
        "latency_s": elapsed,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": completion_tokens,
        "tokens/s": completion_tokens / elapsed
        if completion_tokens is not None
        else None,
        "finish_reason": choice.get("finish_reason"),
        "generated_text": text,
        "reasoning_text": reasoning,
        "status": "OK" if text.strip() else "NO_VISIBLE_TEXT",
        "error": "",
    }

In [4]:
TEST_PROMPT = "Explain attention mechanisms in transformer models in 2-3 sentences."
for model in MODEL_NAMES:
    result = chat(TEST_PROMPT, model)
    print(f"\n{model}: {result['latency_s']:.3f}s ({result['finish_reason']})")
    print(result["generated_text"] or "No visible answer; increase MAX_TOKENS.")


gemma4:26b: 33.643s (stop)
Attention mechanisms allow a Transformer to dynamically weigh the importance of different tokens in a sequence relative to a specific target token. By calculating these relevance scores, the model can capture complex contextual relationships and long-range dependencies, regardless of how far apart the words are in a sentence. This enables the model to build a holistic understanding of the text by focusing on the most relevant information for every part of the input.


## Benchmark
Full prompts and responses are retained. HTTP, decoding, and malformed-response failures are excluded from success statistics. Missing token usage remains unknown. Completion tokens per second includes end-to-end request overhead and any reasoning tokens reported by the server. `NO_VISIBLE_TEXT` indicates a reasoning-only response; increase the token budget if needed. A `length` finish reason indicates truncation.

In [ ]:
if NUM_REQUESTS < 1 or not BENCHMARK_PROMPTS or not MODEL_NAMES:
    raise ValueError("Configure at least one model, prompt, and request")

responses_collected = []
for model in MODEL_NAMES:
    for i in range(NUM_REQUESTS):
        prompt = BENCHMARK_PROMPTS[i % len(BENCHMARK_PROMPTS)]
        try:
            result = chat(prompt, model)
        except (requests.RequestException, ValueError, KeyError, TypeError) as error:
            result = {
                "model": model,
                "served_model": None,
                "latency_s": None,
                "prompt_tokens": None,
                "completion_tokens": None,
                "tokens/s": None,
                "finish_reason": None,
                "generated_text": "",
                "reasoning_text": "",
                "status": "FAILED",
                "error": str(error),
            }
        result.update(request=i + 1, prompt=prompt)
        responses_collected.append(result)
        print(f"{model} {i + 1}/{NUM_REQUESTS}: {result['status']}")

df_results = pd.DataFrame(responses_collected)
display(df_results)
summary_rows = []
for model in MODEL_NAMES:
    rows = df_results[df_results["model"] == model]
    ok = rows[rows["status"] == "OK"]
    # Only measured visible answers enter latency and throughput statistics.
    total_time = ok["latency_s"].sum()
    known_usage = not ok.empty and ok["completion_tokens"].notna().all()
    summary_rows.append(
        {
            "model": model,
            "successful": len(ok),
            "failed": int((rows["status"] == "FAILED").sum()),
            "no_visible_text": int((rows["status"] == "NO_VISIBLE_TEXT").sum()),
            "mean_latency_s": ok["latency_s"].mean(),
            "median_latency_s": ok["latency_s"].median(),
            "min_latency_s": ok["latency_s"].min(),
            "max_latency_s": ok["latency_s"].max(),
            "completion_tokens/s": (
                ok["completion_tokens"].sum() / total_time
                if known_usage and total_time > 0
                else None
            ),
        }
    )
df_summary = pd.DataFrame(summary_rows)
display(df_summary)

gemma4:26b 1/5: OK
gemma4:26b 2/5: OK
gemma4:26b 3/5: OK
gemma4:26b 4/5: OK
gemma4:26b 5/5: OK


,model,served_model,latency_s,prompt_tokens,completion_tokens,tokens/s,finish_reason,generated_text,reasoning_text,status,error,request,prompt
0,gemma4:26b,gemma4:26b,1.477708,23,38,25.715505,stop,The capital of France is **Paris**.,The user is asking for the capital of France.\...,OK,,1,What is the capital of France?
1,gemma4:26b,gemma4:26b,19.618564,24,645,32.877025,stop,"Woven from silicon, patterned in code,\nA trav...",Artificial Intelligence (AI).\nShort poem.\nRe...,OK,,2,Write a short poem about artificial intelligence.
2,gemma4:26b,gemma4:26b,22.523137,23,800,35.519030,length,"To understand quantum computing, you first hav...",\n* Topic: Quantum Computing.\n * Targe...,OK,,3,Explain quantum computing in simple terms.
3,gemma4:26b,gemma4:26b,14.958416,24,575,38.439898,stop,The Three Laws of Robotics were formulated by ...,"* Question: ""What are the three laws of robo...",OK,,4,What are the three laws of robotics?
4,gemma4:26b,gemma4:26b,24.564516,26,800,32.567302,length,"Directed by Christopher Nolan, *Inception* is ...",\n* Subject: Movie *Inception* (directed by ...,OK,,5,Summarize the plot of the movie Inception.


,model,successful,failed,no_visible_text,mean_latency_s,median_latency_s,min_latency_s,max_latency_s,completion_tokens/s
0,gemma4:26b,5,0,0,16.628468,19.618564,1.477708,24.564516,34.374784


## Inspect response text
These descriptive length, vocabulary, and repetition measures do not evaluate factual correctness. They reuse the measured responses without issuing extra requests. Compare notebook runs only with matching prompts, token budgets, temperatures, and server conditions; no historical cluster numbers are inserted.

In [ ]:
def compute_quality_metrics(text):
    words = text.lower().split()
    trigrams = list(zip(words, words[1:], words[2:]))
    counts = Counter(trigrams)
    return {
        "char_len": len(text),
        "word_count": len(words),
        "type_token_ratio": len(set(words)) / len(words) if words else 0.0,
        "repeat_3gram_ratio": (
            sum(count - 1 for count in counts.values()) / len(trigrams)
            if trigrams
            else 0.0
        ),
    }


quality_rows = [
    {
        "model": row["model"],
        "request": row["request"],
        "prompt": row["prompt"],
        **compute_quality_metrics(row["generated_text"]),
    }
    for row in responses_collected
    if row["status"] == "OK"
]
df_quality = pd.DataFrame(quality_rows)
if df_quality.empty:
    print("No successful visible responses to inspect. See df_results for errors.")
else:
    display(df_quality)
    display(df_quality.groupby("model").mean(numeric_only=True).drop(columns="request"))
    for row in responses_collected:
        if row["status"] == "OK":
            print(f"\n[{row['model']}] {row['prompt']}")
            print(row["generated_text"])

,model,request,prompt,char_len,word_count,type_token_ratio,repeat_3gram_ratio
0,gemma4:26b,1,What is the capital of France?,35,6,1.000000,0.000000
1,gemma4:26b,2,Write a short poem about artificial intelligence.,342,61,0.737705,0.000000
2,gemma4:26b,3,Explain quantum computing in simple terms.,560,97,0.731959,0.010526
3,gemma4:26b,4,What are the three laws of robotics?,812,141,0.652482,0.079137
4,gemma4:26b,5,Summarize the plot of the movie Inception.,396,64,0.812500,0.000000


,char_len,word_count,type_token_ratio,repeat_3gram_ratio
model,,,,
gemma4:26b,429.0,73.8,0.786929,0.017933



[gemma4:26b] What is the capital of France?
The capital of France is **Paris**.

[gemma4:26b] Write a short poem about artificial intelligence.
Woven from silicon, patterned in code,
A traveler walking an infinite road.
It drinks from the rivers of data and lore,
Knocking on wisdom’s unlatched door.

No heartbeat to quicken, no breath in the chest,
Yet it mirrors the mind at a human request.
A spark in the circuit, a ghost in the stream—
The waking reflection of a collective dream.

[gemma4:26b] Explain quantum computing in simple terms.
To understand quantum computing, you first have to understand how a normal computer (like your phone or laptop) works.

### 1. The Old Way: Classical Computing
Every piece of information in a normal computer is stored in **"bits."** A bit is like a tiny light switch that can be in one of two states: **On or Off** (represented as **1 or 0**).

Everything you do—typing an email, watching a YouTube video, playing a game—is just the result of billions of 